In [ ]:
import kagglehub

path = kagglehub.dataset_download("hungle3401/faceforensics")
print("Path to dataset files:", path)

Using Colab cache for faster access to the 'faceforensics' dataset.
Path to dataset files: /kaggle/input/faceforensics


In [ ]:
!pip install facenet-pytorch opencv-python

In [ ]:
import os
import cv2
import torch
import random
from tqdm import tqdm
from facenet_pytorch import MTCNN

# =========================
# CONFIG
# =========================
input_root = "/kaggle/input/faceforensics/FF++"   # fake/ real/
output_root = "/content/processed_ffpp"
img_size = 224
frame_skip = 10
margin_ratio = 0.25
seed = 42

classes = ["fake", "real"]

# =========================
# REPRODUCIBILITY
# =========================
random.seed(seed)

# =========================
# FACE DETECTOR
# =========================
device = "cuda" if torch.cuda.is_available() else "cpu"
mtcnn = MTCNN(keep_all=False, device=device)

# =========================
# HELPERS
# =========================
def is_video_file(filename):
    return filename.lower().endswith((".mp4", ".avi", ".mov", ".mkv"))

def make_split(video_list, train_ratio=0.7, val_ratio=0.15):
    random.shuffle(video_list)
    n = len(video_list)
    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)
    train_videos = video_list[:n_train]
    val_videos = video_list[n_train:n_train + n_val]
    test_videos = video_list[n_train + n_val:]
    return train_videos, val_videos, test_videos

def expand_box(x1, y1, x2, y2, w, h, margin_ratio=0.25):
    bw = x2 - x1
    bh = y2 - y1
    mx = int(bw * margin_ratio)
    my = int(bh * margin_ratio)

    x1 = max(0, x1 - mx)
    y1 = max(0, y1 - my)
    x2 = min(w, x2 + mx)
    y2 = min(h, y2 + my)
    return x1, y1, x2, y2

def extract_faces_from_video(video_path, save_dir):
    os.makedirs(save_dir, exist_ok=True)

    cap = cv2.VideoCapture(video_path)
    frame_id = 0
    saved_id = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_id % frame_skip == 0:
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

            boxes, _ = mtcnn.detect(rgb)

            if boxes is not None and len(boxes) > 0:
                # use first / largest detected face
                x1, y1, x2, y2 = boxes[0]
                x1, y1, x2, y2 = map(int, [x1, y1, x2, y2])

                h, w, _ = rgb.shape
                x1, y1, x2, y2 = expand_box(x1, y1, x2, y2, w, h, margin_ratio)

                face = rgb[y1:y2, x1:x2]

                if face.size > 0:
                    face = cv2.resize(face, (img_size, img_size))
                    out_path = os.path.join(save_dir, f"{saved_id:04d}.jpg")
                    cv2.imwrite(out_path, cv2.cvtColor(face, cv2.COLOR_RGB2BGR))
                    saved_id += 1

        frame_id += 1

    cap.release()

# =========================
# SPLIT + PREPROCESS
# =========================
splits = ["train", "val", "test"]
for split in splits:
    for cls in classes:
        os.makedirs(os.path.join(output_root, split, cls), exist_ok=True)

for cls in classes:
    input_dir = os.path.join(input_root, cls)
    videos = [v for v in os.listdir(input_dir) if is_video_file(v)]

    train_videos, val_videos, test_videos = make_split(videos)

    split_map = {
        "train": train_videos,
        "val": val_videos,
        "test": test_videos
    }

    for split_name, split_videos in split_map.items():
        for video_name in tqdm(split_videos, desc=f"{cls}-{split_name}"):
            video_path = os.path.join(input_dir, video_name)
            video_id = os.path.splitext(video_name)[0]
            save_dir = os.path.join(output_root, split_name, cls, video_id)
            extract_faces_from_video(video_path, save_dir)

print("Preprocessing complete.")

fake-train:  54%|█████▍    | 76/140 [1:29:50<1:29:22, 83.79s/it]

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!cp -r /content/processed_ffpp /content/drive/MyDrive/